In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# Διαδρομές των τελικών datasets
train_path = Path("processed/train_final_scaled.csv")
test_path = Path("processed/test_final_scaled.csv")

# Έλεγχος ύπαρξης αρχείων
assert train_path.exists(), "Training file not found!"
assert test_path.exists(), "Testing file not found!"

# Διαβάζουμε τα ονόματα των χαρακτηριστικών
columns = pd.read_csv(train_path, nrows=0).columns.tolist()
features = [col for col in columns if col != "Label"]

# float32 για χαμηλότερη χρήση RAM
dtypes = {col: "float32" for col in features}
dtypes["Label"] = "int16"

# Φόρτωση δεδομένων
train_df = pd.read_csv(train_path, dtype=dtypes)
test_df = pd.read_csv(test_path, dtype=dtypes)

# Βασικοί έλεγχοι
assert train_df.columns.tolist() == test_df.columns.tolist()
assert np.isfinite(train_df[features].to_numpy()).all()
assert np.isfinite(test_df[features].to_numpy()).all()

print("=== DATASET INFORMATION ===")
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)
print("Number of features:", len(features))

print("\n=== TRAINING LABELS ===")
print(train_df["Label"].value_counts().sort_index())

print("\n=== TESTING LABELS ===")
print(test_df["Label"].value_counts().sort_index())

print("\n=== MEMORY USAGE ===")
print(
    "Train:",
    round(train_df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)
print(
    "Test:",
    round(test_df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)


In [ ]:

import numpy as np
from sklearn.model_selection import train_test_split

# Αρχική ετικέτα φυσιολογικής κίνησης
BENIGN_LABEL = 1

# Μετατροπή σε binary classification
# 0 = Benign, 1 = Attack
y_full = (
    train_df["Label"].to_numpy() != BENIGN_LABEL
).astype(np.float32)

y_test = (
    test_df["Label"].to_numpy() != BENIGN_LABEL
).astype(np.float32)

# Δημιουργία Training / Validation split
train_idx, val_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=0.15,
    random_state=42,
    stratify=y_full
)

# Χαρακτηριστικά
X_full = train_df[features].to_numpy(dtype=np.float32)

X_train = X_full[train_idx]
X_val = X_full[val_idx]
X_test = test_df[features].to_numpy(dtype=np.float32)

# Ετικέτες
y_train = y_full[train_idx]
y_val = y_full[val_idx]

# Έλεγχοι
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

assert X_train.shape[1] == 44
assert X_val.shape[1] == 44
assert X_test.shape[1] == 44

print("=== FINAL DATA SPLITS ===")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\n=== CLASS DISTRIBUTION ===")

for name, labels in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Testing", y_test)
]:
    benign = int((labels == 0).sum())
    attacks = int((labels == 1).sum())

    print(f"\n{name}:")
    print("Benign:", benign)
    print("Attack:", attacks)
    print("Attack percentage:", round(100 * attacks / len(labels), 2), "%")

print("\nData preparation completed!")


In [ ]:

import sys

print("Python environment:")
print(sys.executable)

try:
    import torch

    print("\nPyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print("Selected device:", device)

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

    print("\nPyTorch is ready!")

except ImportError:
    print("\nPyTorch is not installed.")
    print("Install it using: %pip install torch")


In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Για αναπαραγωγιμότητα
torch.manual_seed(42)

# 1. Μετατροπή των δεδομένων σε PyTorch tensors
train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train).reshape(-1, 1)
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(y_val).reshape(-1, 1)
)

# 2. DataLoaders
BATCH_SIZE = 512

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# 3. Ορισμός νευρωνικού δικτύου
class IntrusionDetectionMLP(nn.Module):

    def __init__(self, input_features=44):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

# 4. Δημιουργία μοντέλου
model = IntrusionDetectionMLP(
    input_features=X_train.shape[1]
).to(device)

# 5. Αντιμετώπιση class imbalance
benign_count = int((y_train == 0).sum())
attack_count = int((y_train == 1).sum())

positive_weight = benign_count / attack_count

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [positive_weight],
        dtype=torch.float32,
        device=device
    )
)

# 6. Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# 7. Έλεγχος
total_params = sum(
    p.numel() for p in model.parameters()
)

print("=== MODEL INFORMATION ===")
print(model)

print("\nDevice:", device)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Trainable parameters:", total_params)
print("Positive class weight:", round(positive_weight, 4))

print("\nModel initialized successfully!")


In [ ]:

import torch
import numpy as np
import time
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score

# Ρυθμίσεις εκπαίδευσης
EPOCHS = 15
PATIENCE = 4

# Φάκελος αποθήκευσης μοντέλων
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

model_path = model_dir / "mlp_binary_best.pt"

# Ιστορικό εκπαίδευσης
history = []

best_val_f1 = -1.0
epochs_without_improvement = 0

print("=== STARTING TRAINING ===")

for epoch in range(EPOCHS):

    start_time = time.perf_counter()

    # -------------------------------
    # 1. TRAINING PHASE
    # -------------------------------
    model.train()

    running_loss = 0.0
    total_samples = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(X_batch)
        total_samples += len(X_batch)

    train_loss = running_loss / total_samples

    # -------------------------------
    # 2. VALIDATION PHASE
    # -------------------------------
    model.eval()

    val_preds = []
    val_targets = []

    with torch.inference_mode():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)

            logits = model(X_batch)

            # Logit >= 0 ισοδυναμεί με
            # sigmoid(logit) >= 0.5
            predictions = (
                logits >= 0
            ).int().cpu().numpy().ravel()

            targets = y_batch.numpy().ravel()

            val_preds.append(predictions)
            val_targets.append(targets)

    val_preds = np.concatenate(val_preds)
    val_targets = np.concatenate(val_targets)

    val_precision = precision_score(
        val_targets, val_preds, zero_division=0
    )

    val_recall = recall_score(
        val_targets, val_preds, zero_division=0
    )

    val_f1 = f1_score(
        val_targets, val_preds, zero_division=0
    )

    epoch_time = time.perf_counter() - start_time

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_precision": val_precision,
        "val_recall": val_recall,
        "val_f1": val_f1,
        "epoch_time": epoch_time
    })

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Precision: {val_precision:.4f} | "
        f"Val Recall: {val_recall:.4f} | "
        f"Val F1: {val_f1:.4f} | "
        f"Time: {epoch_time:.1f}s",
        flush=True
    )

    # -------------------------------
    # 3. SAVE BEST MODEL
    # -------------------------------
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            model_path
        )

        print("  Best model saved!", flush=True)

    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping triggered!")
            break

# Φορτώνουμε την καλύτερη έκδοση
model.load_state_dict(
    torch.load(
        model_path,
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("\n=== TRAINING COMPLETED ===")
print("Epochs completed:", len(history))
print("Best Validation F1:", round(best_val_f1, 4))
print("Best model:", model_path)


In [ ]:

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# 1. Φόρτωση του καλύτερου μοντέλου
model_path = Path("models/mlp_binary_best.pt")

model.load_state_dict(
    torch.load(
        model_path,
        map_location=device,
        weights_only=True
    )
)

model.eval()

# 2. Μετατροπή του Testing Set σε tensor
X_test_tensor = torch.from_numpy(X_test)

test_predictions = []

# 3. Πρόβλεψη σε batches
BATCH_SIZE_TEST = 512

with torch.inference_mode():

    for start in range(
        0, len(X_test_tensor), BATCH_SIZE_TEST
    ):

        batch = X_test_tensor[
            start:start + BATCH_SIZE_TEST
        ].to(device)

        logits = model(batch)

        predictions = (
            logits >= 0
        ).int().cpu().numpy().ravel()

        test_predictions.append(predictions)

y_pred = np.concatenate(test_predictions)
y_true = y_test.astype(int)

# 4. Υπολογισμός μετρικών
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(
    y_true, y_pred, zero_division=0
)
recall = recall_score(
    y_true, y_pred, zero_division=0
)
f1 = f1_score(
    y_true, y_pred, zero_division=0
)

cm = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn) if (fp + tn) else 0.0
specificity = tn / (tn + fp) if (tn + fp) else 0.0

# 5. Εμφάνιση αποτελεσμάτων
print("=== TEST SET RESULTS ===")
print(f"Accuracy:    {accuracy:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-score:    {f1:.4f}")
print(f"FPR:         {fpr:.6f}")
print(f"Specificity: {specificity:.4f}")

print("\n=== CONFUSION MATRIX ===")
print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

print("\n=== CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Benign", "Attack"],
        zero_division=0
    )
)

# 6. Confusion Matrix Heatmap
Path("results").mkdir(exist_ok=True)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Benign", "Attack"],
    yticklabels=["Benign", "Attack"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("MLP - Test Confusion Matrix")
plt.tight_layout()

plt.savefig(
    "results/mlp_confusion_matrix.png",
    dpi=150
)

plt.show()

# 7. Αποθήκευση των μετρικών
metrics = pd.DataFrame([{
    "Model": "MLP",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "FPR": fpr,
    "Specificity": specificity,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}])

metrics.to_csv(
    "results/mlp_test_metrics.csv",
    index=False
)

print("\nEvaluation completed!")


In [ ]:

import torch
import numpy as np
import time
import json
from pathlib import Path

# Φόρτωση καλύτερου μοντέλου
model.load_state_dict(
    torch.load(
        "models/mlp_binary_best.pt",
        map_location=device,
        weights_only=True
    )
)

model.eval()

# Τα δεδομένα βρίσκονται ήδη στη μνήμη
X_benchmark = torch.from_numpy(
    np.ascontiguousarray(X_test, dtype=np.float32)
).to(device)

num_samples = len(X_benchmark)
batch_size = 512

print("=== INFERENCE BENCHMARK ===")
print("Device:", device)
print("Samples:", num_samples)
print("CPU threads:", torch.get_num_threads())

# =======================================
# 1. WARM-UP
# =======================================

with torch.inference_mode():
    for _ in range(100):
        _ = model(X_benchmark[:1])

    for _ in range(20):
        _ = model(X_benchmark[:batch_size])

print("\nWarm-up completed!")

# =======================================
# 2. SINGLE-FLOW LATENCY
# =======================================

single_times_ms = []

with torch.inference_mode():
    for i in range(2000):

        sample = X_benchmark[i:i+1]

        start = time.perf_counter_ns()
        _ = model(sample)
        end = time.perf_counter_ns()

        single_times_ms.append(
            (end - start) / 1_000_000
        )

single_mean = np.mean(single_times_ms)
single_median = np.median(single_times_ms)
single_p95 = np.percentile(single_times_ms, 95)

print("\n=== SINGLE-FLOW LATENCY ===")
print(f"Mean:   {single_mean:.6f} ms")
print(f"Median: {single_median:.6f} ms")
print(f"P95:    {single_p95:.6f} ms")

# =======================================
# 3. BATCH INFERENCE THROUGHPUT
# =======================================

run_times_seconds = []

with torch.inference_mode():
    for repetition in range(5):

        start = time.perf_counter()

        for i in range(0, num_samples, batch_size):
            batch = X_benchmark[i:i+batch_size]
            _ = model(batch)

        elapsed = time.perf_counter() - start
        run_times_seconds.append(elapsed)

median_total_time = float(
    np.median(run_times_seconds)
)

throughput = num_samples / median_total_time

# Μέσος χρόνος ανά ροή όταν
# επεξεργαζόμαστε δεδομένα σε batches
amortized_latency_ms = (
    median_total_time / num_samples
) * 1000

print("\n=== BATCH INFERENCE ===")
print("Batch size:", batch_size)
print(f"Median full-test time: {median_total_time:.4f} s")
print(f"Throughput: {throughput:.2f} flows/second")
print(f"Amortized time per flow: {amortized_latency_ms:.6f} ms")

# =======================================
# 4. SAVE RESULTS
# =======================================

results = {
    "model": "MLP",
    "device": str(device),
    "torch_version": torch.__version__,
    "cpu_threads": torch.get_num_threads(),
    "test_samples": num_samples,
    "single_flow_mean_ms": float(single_mean),
    "single_flow_median_ms": float(single_median),
    "single_flow_p95_ms": float(single_p95),
    "batch_size": batch_size,
    "batch_total_median_seconds": median_total_time,
    "throughput_flows_per_second": float(throughput),
    "batch_amortized_ms_per_flow": float(amortized_latency_ms)
}

Path("results").mkdir(exist_ok=True)

with open(
    "results/mlp_inference_results.json",
    "w"
) as f:
    json.dump(results, f, indent=4)

print("\nInference benchmark completed!")


In [ ]:

import pandas as pd
import json
from pathlib import Path

results_dir = Path("results")

# Μετρικές ταξινόμησης
classification = pd.read_csv(
    results_dir / "mlp_test_metrics.csv"
).iloc[0]

# Μετρικές ταχύτητας
with open(
    results_dir / "mlp_inference_results.json",
    "r"
) as f:
    benchmark = json.load(f)

# Μέγεθος αποθηκευμένου μοντέλου
model_path = Path("models/mlp_binary_best.pt")
model_size_kb = model_path.stat().st_size / 1024

# Συγκεντρωτικός πίνακας
summary = pd.DataFrame([{
    "Model": "MLP",
    "Accuracy": classification["Accuracy"],
    "Precision": classification["Precision"],
    "Recall": classification["Recall"],
    "F1": classification["F1"],
    "FPR": classification["FPR"],
    "Single Mean Latency (ms)": benchmark["single_flow_mean_ms"],
    "Single Median Latency (ms)": benchmark["single_flow_median_ms"],
    "Single P95 Latency (ms)": benchmark["single_flow_p95_ms"],
    "Batch Size": benchmark["batch_size"],
    "Throughput (flows/s)": benchmark["throughput_flows_per_second"],
    "Model Size (KB)": model_size_kb,
    "Device": benchmark["device"],
    "CPU Threads": benchmark["cpu_threads"]
}])

summary.to_csv(
    results_dir / "mlp_final_summary.csv",
    index=False
)

print("=== MLP FINAL SUMMARY ===")
print(summary.T.to_string(header=False))

print("\nSaved: results/mlp_final_summary.csv")
